In [1]:
# ============================================
# YOLOv12 Training in Google Colab
# ============================================

# 1. Install required packages (run this first)
!pip install ultralytics pyyaml -q
print("✅ Packages installed")

# 2. Import libraries
import os
import zipfile
import yaml
import time
import torch
from google.colab import files
from ultralytics import YOLO

# ============================================
# 3. Upload dataset zip
# ============================================
print("\n📤 Please upload your dataset zip file...")
uploaded = files.upload()

for filename in uploaded.keys():
    size_mb = len(uploaded[filename]) / (1024 * 1024)
    print(f'Uploaded: {filename} ({size_mb:.2f} MB)')

# ============================================
# 4. Locate and extract zip
# ============================================
zip_files = [f for f in os.listdir('/content') if f.endswith('.zip')]
if not zip_files:
    zip_filename = list(uploaded.keys())[0]
else:
    zip_filename = zip_files[0]
print(f"\n📦 Using zip file: {zip_filename}")

extract_base = "/content/datasets"
dataset_folder = os.path.join(extract_base, "traffic_dataset")
if os.path.exists(dataset_folder):
    !rm -rf "{dataset_folder}"
os.makedirs(dataset_folder, exist_ok=True)

print(f"Extracting to {dataset_folder}...")
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(dataset_folder)

# ============================================
# 5. Display folder structure (for debugging)
# ============================================
print("\n📁 Extracted folder structure (first 3 levels):")
for root, dirs, files in os.walk(dataset_folder):
    level = root.replace(dataset_folder, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        for i, f in enumerate(files):
            if i < 10:
                print(f"{indent}  {f}")
            else:
                print(f"{indent}  ... (and {len(files)-10} more)")
                break

# ============================================
# 6. Find or create data.yaml
# ============================================
def find_data_yaml(base_path):
    for root, _, files in os.walk(base_path):
        if 'data.yaml' in files:
            return os.path.join(root, 'data.yaml')
    return None

yaml_file = find_data_yaml(dataset_folder)

if yaml_file is None:
    print("\n⚠️ data.yaml not found. Attempting to auto-create one...")

    # Helper to find image folders
    def find_folder(base, target):
        for root, dirs, _ in os.walk(base):
            if target in dirs:
                # Check if it contains images
                candidate = os.path.join(root, target)
                # Look for image files
                for f in os.listdir(candidate):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        return candidate
        return None

    train_img_folder = find_folder(dataset_folder, 'train') or find_folder(dataset_folder, 'images')
    val_img_folder = find_folder(dataset_folder, 'valid') or find_folder(dataset_folder, 'val')
    test_img_folder = find_folder(dataset_folder, 'test')

    if train_img_folder and val_img_folder:
        # Try to find classes.txt
        nc = None
        names = None
        classes_txt = None
        for root, _, files in os.walk(dataset_folder):
            if 'classes.txt' in files:
                classes_txt = os.path.join(root, 'classes.txt')
                break
        if classes_txt:
            with open(classes_txt, 'r') as f:
                names = [line.strip() for line in f.readlines() if line.strip()]
            nc = len(names)
        else:
            # Default placeholder
            nc = 1
            names = ['object']
            print("⚠️ No classes.txt found. Using placeholder 'object'. You may need to edit data.yaml later.")

        data_config = {
            'train': train_img_folder.replace('\\', '/'),
            'val': val_img_folder.replace('\\', '/'),
            'nc': nc,
            'names': names
        }
        if test_img_folder:
            data_config['test'] = test_img_folder.replace('\\', '/')

        yaml_file = os.path.join(dataset_folder, 'data.yaml')
        with open(yaml_file, 'w') as f:
            yaml.dump(data_config, f, default_flow_style=False)
        print(f"✅ Auto-generated data.yaml at {yaml_file}")
        print(f"Classes: {names}")
    else:
        print("\n❌ Could not auto-detect dataset structure. Manual input required.")
        print("Please enter the paths based on the folder structure shown above.")
        train_path = input("Enter full path to train images folder: ").strip()
        val_path = input("Enter full path to val images folder: ").strip()
        nc = int(input("Number of classes: "))
        names_input = input("Class names (comma-separated, e.g., car,person,bicycle): ").strip()
        names = [n.strip() for n in names_input.split(',')]
        data_config = {
            'train': train_path,
            'val': val_path,
            'nc': nc,
            'names': names
        }
        test_path = input("Enter test images folder (optional, press Enter to skip): ").strip()
        if test_path:
            data_config['test'] = test_path
        yaml_file = os.path.join(dataset_folder, 'data.yaml')
        with open(yaml_file, 'w') as f:
            yaml.dump(data_config, f, default_flow_style=False)
        print(f"✅ Saved custom data.yaml at {yaml_file}")

# ============================================
# 7. Verify YAML content
# ============================================
print("\n📄 data.yaml content:")
with open(yaml_file, 'r') as f:
    print(f.read())

# ============================================
# 8. Check GPU
# ============================================
print(f"\n🖥️ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# ============================================
# 9. Train YOLOv12
# ============================================
print("\n🚀 Starting YOLOv12 training...")
start_time = time.time()

# Choose YOLOv12 variant: yolo12n.pt (nano), yolo12s.pt (small), yolo12m.pt (medium), yolo12l.pt (large), yolo12x.pt (xlarge)
model = YOLO('yolo12m.pt')

results = model.train(
    data=yaml_file,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,
    project='/content/runs',
    name='traffic_100epochs',
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
)

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Training completed in {elapsed:.2f} hours.")

# ============================================
# 10. Evaluate best model
# ============================================
best_model_path = '/content/runs/traffic_100epochs/weights/best.pt'
if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    metrics = best_model.val()

    print("\n📊 Validation Results:")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall: {metrics.box.mr:.4f}")
    print(f"   mAP@0.5: {metrics.box.map50:.4f}")
    print(f"   mAP@0.5:0.95: {metrics.box.map:.4f}")

    print("\n📈 Per-class metrics:")
    for i, class_name in enumerate(metrics.names.values()):
        if i < len(metrics.box.p):
            print(f"   {class_name}:")
            print(f"     Precision: {metrics.box.p[i]:.4f}")
            print(f"     Recall: {metrics.box.r[i]:.4f}")
            print(f"     mAP50: {metrics.box.ap50[i]:.4f}")
else:
    print(f"\n❌ Best model not found at {best_model_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.4 MB/s eta 0:00:00
✅ Packages installed
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

📤 Please upload your dataset zip file...


Saving Wireless Int Modified.v5i.retinanet.zip to Wireless Int Modified.v5i.retinanet.zip
Uploaded: Wireless Int Modified.v5i.retinanet.zip (11.61 MB)

📦 Using zip file: Wireless Int Modified.v5i.retinanet.zip
Extracting to /content/datasets/traffic_dataset...

📁 Extracted folder structure (first 3 levels):
traffic_dataset/
  README.roboflow.txt
  README.dataset.txt
  valid/
    0217_mp4-0001_jpg.rf.64e46da853e94b8ddab25dbac4db74e9.jpg
    0217_mp4-0002_jpg.rf.e75cdfae21ac26316868cd47d4376f33.jpg
    0217_mp4-0008_jpg.rf.c819c8c93d0f0457b53cd32e02eef2b9.jpg
    0217_mp4-0021_jpg.rf.9cb32862fa43a66fc61b9c7f561dca2a.jpg
    0217_mp4-0015_jpg.rf.ccd7e49f5abe72ed7344d9c5a28646ff.jpg
    0217_mp4-0004_jpg.rf.8ab9c7609b68d92e33a17da72722868b.jpg
    _annotations.csv
  test/
    0217_mp4-0035_jpg.rf.d8cac64c314780c15a02e925077590b6.jpg
    0217_mp4-0010_jpg.rf.bf02dee61c8eb4c2de7574bc5f625944.jpg
    0217_mp4-0024_jpg.rf.a2ac565c7c33a0992dfe43e810a3c14c.jpg
    0217_mp4-0012_jpg.rf.d2efbefa08

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100        11G          0      174.7          0          0        640: 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 8.4it/s 0.1s
                   all          6          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


...


100 epochs completed in 0.101 hours.
Optimizer stripped from /content/runs/traffic_100epochs/weights/last.pt, 40.7MB
Optimizer stripped from /content/runs/traffic_100epochs/weights/best.pt, 40.7MB

Validating /content/runs/traffic_100epochs/weights/best.pt...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 13.8it/s 0.1s


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:657: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all          6          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 0.1ms preprocess, 10.7ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/traffic_100epochs

✅ Training completed in 0.11 hours.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4455.3±1087.5 MB/s, size: 312.0 KB)
val: Scanning /content/datasets/traffic_dataset/valid.cache... 0 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6/6 2.8Mit/s 0.0s
WARNING ⚠️ Labels are missing or empty in /content/datasets/traffic_dataset/valid.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:657: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all          6          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 0.5ms preprocess, 25.4ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /content/runs/detect/val

📊 Validation Results:
   Precision: 0.0000
   Recall: 0.0000
   mAP@0.5: 0.0000
   mAP@0.5:0.95: 0.0000

📈 Per-class metrics:
